# Capítulo 2 - Exercício 5: Explorar opções de preparação com `GridSearchCV`

Neste exercício usamos `GridSearchCV` para testar automaticamente algumas opções do pipeline. O foco é a etapa geográfica criada no exercício anterior: um transformador personalizado que usa `latitude` e `longitude` para gerar uma característica baseada em `KNeighborsRegressor`.

## Plano do exercício

1. Carregar os dados e recriar a divisão estratificada.
2. Recriar o pré-processamento base do projeto.
3. Criar o transformador `FeatureFromRegressor`.
4. Substituir a etapa `geo` por um `KNeighborsRegressor` usado como transformador.
5. Montar um pipeline com o novo pré-processamento e um `SVR`.
6. Usar `GridSearchCV` para testar opções de preparação e alguns parâmetros do modelo.
7. Avaliar a melhor configuração encontrada.

## Configuração

Importamos as bibliotecas usadas no exercício e fixamos a semente aleatória para tornar os resultados mais reprodutíveis.

In [1]:
import sys
from pathlib import Path
import tarfile
import urllib.request

import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

## Carregando os dados

Esta função baixa e extrai o conjunto de dados se ele ainda não estiver disponível localmente.

In [2]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
        housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

/tmp/ipykernel_16125/2931591054.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Divisão treino/teste estratificada

Mantemos a mesma estratégia do capítulo: criar categorias de renda para preservar melhor a distribuição de `median_income` no treino e no teste.

In [3]:
from sklearn.model_selection import train_test_split

housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing,
    test_size=0.2,
    stratify=housing["income_cat"],
    random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

## Pré-processamento base

Este é o pipeline de preparação usado no projeto principal. Depois vamos trocar apenas a etapa `geo`, mantendo as demais transformações iguais.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Similaridade com o cluster {i}" for i in range(self.n_clusters)]


log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                           "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

## Métrica de avaliação

Usaremos RMSE, a mesma métrica do projeto principal. O Scikit-Learn usa a versão negativa da métrica durante a busca, então depois multiplicamos por `-1` para ler o erro real.

In [5]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

## Transformador personalizado

Este transformador encapsula um regressor. No `fit()`, ele treina o regressor usando as entradas e o alvo. No `transform()`, ele devolve as predições como uma nova coluna de atributos.

In [6]:
from sklearn.base import MetaEstimatorMixin, clone
from sklearn.neighbors import KNeighborsRegressor
from sklearn.utils.validation import check_array, check_is_fitted


class FeatureFromRegressor(MetaEstimatorMixin, TransformerMixin, BaseEstimator):
    def __init__(self, estimator):
        self.estimator = estimator

    def fit(self, X, y=None):
        X = check_array(X)
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        self.n_features_in_ = self.estimator_.n_features_in_
        if hasattr(self.estimator_, "feature_names_in_"):
            self.feature_names_in_ = self.estimator_.feature_names_in_
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        predictions = self.estimator_.predict(X)
        if predictions.ndim == 1:
            predictions = predictions.reshape(-1, 1)
        return predictions

    def get_feature_names_out(self, names=None):
        check_is_fitted(self)
        n_outputs = getattr(self.estimator_, "n_outputs_", 1)
        estimator_class_name = self.estimator_.__class__.__name__
        estimator_short_name = estimator_class_name.lower().replace("_", "")
        return [f"{estimator_short_name}_prediction_{i}"
                for i in range(n_outputs)]

## Nova etapa geográfica

Aqui substituímos a etapa `geo` original. Em vez de similaridade com clusters, ela passa a usar `KNeighborsRegressor` para estimar o preço mediano com base em `latitude` e `longitude`.

In [7]:
knn_reg = KNeighborsRegressor(n_neighbors=3, weights="distance")
knn_transformer = FeatureFromRegressor(knn_reg)

transformers = [(name, clone(transformer), columns)
                for name, transformer, columns in preprocessing.transformers]
geo_index = [name for name, _, _ in transformers].index("geo")
transformers[geo_index] = ("geo", knn_transformer, ["latitude", "longitude"])

new_geo_preprocessing = ColumnTransformer(transformers)

## Pipeline que será otimizado

O pipeline completo contém o novo pré-processamento e um `SVR`. Como tudo está dentro do pipeline, o `GridSearchCV` consegue alterar parâmetros internos usando nomes compostos com `__`.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

new_geo_pipeline = Pipeline([
    ("preprocessing", new_geo_preprocessing),
    ("svr", SVR()),
])

## Busca em grade

A busca abaixo testa opções de preparação e alguns parâmetros do `SVR`:

- `preprocessing__geo__estimator__n_neighbors`: quantidade de vizinhos usados pela característica geográfica.
- `preprocessing__geo__estimator__weights`: peso uniforme ou ponderado pela distância.
- `svr__C`, `svr__gamma` e `svr__kernel`: parâmetros do regressor final.

A grade está propositalmente pequena para o exercício não ficar pesado demais. Você pode aumentar as listas depois que a estrutura estiver funcionando.

In [9]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "preprocessing__geo__estimator__n_neighbors": [1, 3, 5, 10],
    "preprocessing__geo__estimator__weights": ["uniform", "distance"],
    "svr__kernel": ["rbf"],
    "svr__C": [10_000, 30_000, 100_000],
    "svr__gamma": [0.03, 0.1, 0.3],
}

new_geo_grid_search = GridSearchCV(
    new_geo_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=2,
)

new_geo_grid_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weights=uniform, svr__C=10000, svr__gamma=0.03, svr__kernel=rbf; total time=   0.8s
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weights=uniform, svr__C=10000, svr__gamma=0.1, svr__kernel=rbf; total time=   0.8s
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weights=uniform, svr__C=10000, svr__gamma=0.3, svr__kernel=rbf; total time=   0.8s
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weights=uniform, svr__C=10000, svr__gamma=0.3, svr__kernel=rbf; total time=   0.8s
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weights=uniform, svr__C=10000, svr__gamma=0.1, svr__kernel=rbf; total time=   0.8s
[CV] END preprocessing__geo__estimator__n_neighbors=1, preprocessing__geo__estimator__weight

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('bedrooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<function ratio_name at 0x7ff15608e340>,
                                                                                                              func=<function column_ratio at 0x7ff15608e0c0>)),
                                                                                         ('standardscaler',
                                                                                          StandardScaler()...
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7ff154ec4530>)])),
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid={'preprocessing__geo__estimator__n_neighbors': [1, 3, 5,
                                                                        10],
                         'preprocessing__geo__estimator__weights': ['uniform',
                                                                    'distance'],
                         'svr__C': [10000, 30000, 100000],
                         'svr__gamma': [0.03, 0.1, 0.3],
                         'svr__kernel': ['rbf']},
             scoring='neg_root_mean_squared_error', verbose=2)

## Resultados da busca

Organizamos os melhores resultados da grade. A coluna `mean_test_rmse` mostra o erro médio de validação cruzada, já convertido para valor positivo.

In [10]:
grid_results = pd.DataFrame(new_geo_grid_search.cv_results_)
grid_results["mean_test_rmse"] = -grid_results["mean_test_score"]

cols = [
    "mean_test_rmse",
    "param_preprocessing__geo__estimator__n_neighbors",
    "param_preprocessing__geo__estimator__weights",
    "param_svr__C",
    "param_svr__gamma",
]

grid_results[cols].sort_values("mean_test_rmse").head(10)

,mean_test_rmse,param_preprocessing__geo__estimator__n_neighbors,param_preprocessing__geo__estimator__weights,param_svr__C,param_svr__gamma
15,72448.357767,1,distance,100000,0.03
6,72448.357767,1,uniform,100000,0.03
16,76346.490330,1,distance,100000,0.10
7,76346.490330,1,uniform,100000,0.10
17,84864.184757,1,distance,100000,0.30
8,84864.184757,1,uniform,100000,0.30
12,86433.501579,1,distance,30000,0.03
3,86433.501579,1,uniform,30000,0.03
24,89787.833877,3,uniform,100000,0.03
13,89922.883152,1,distance,30000,0.10


In [11]:
new_geo_grid_search.best_params_, -new_geo_grid_search.best_score_

({'preprocessing__geo__estimator__n_neighbors': 1,
  'preprocessing__geo__estimator__weights': 'distance',
  'svr__C': 100000,
  'svr__gamma': 0.03,
  'svr__kernel': 'rbf'},
 np.float64(72448.35776713802))

## Avaliação no conjunto de teste

Depois de escolher a melhor combinação pela validação cruzada, avaliamos o melhor pipeline uma única vez no conjunto de teste.

In [12]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

best_new_geo_model = new_geo_grid_search.best_estimator_
new_geo_test_predictions = best_new_geo_model.predict(X_test)
new_geo_test_rmse = root_mean_squared_error(y_test, new_geo_test_predictions)

new_geo_test_rmse

np.float64(70494.80373852846)

## Observação

Se a nova característica geográfica com KNN não superar a similaridade por clusters, isso não significa que o exercício falhou. O objetivo aqui é aprender a expor opções de preparação dentro do pipeline e deixá-las ajustáveis pelo `GridSearchCV`.